In [ ]:
# 1. Install required eval packages (Run this in your terminal first):
# pip install ragas datasets

import sys
import os
import pandas as pd
from datasets import Dataset

sys.path.append(os.path.abspath('..'))
from src.main import process_query
from src import config

# --- Setup Evaluator Models (Using free Groq & HF instead of OpenAI) ---
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy

eval_llm = ChatGroq(model=config.LLM_MODEL, temperature=0)
eval_embeddings = HuggingFaceEmbeddings(
    model_name=config.EMBEDDING_MODEL,
    model_kwargs={'device': 'cpu'}
)

ragas_llm = LangchainLLMWrapper(eval_llm)
ragas_emb = LangchainEmbeddingsWrapper(eval_embeddings)

# --- 2. Create the Evaluation Dataset ---
# For the sake of time/rate limits, we test 3 questions. Add up to 20 for your final resume!
eval_questions = [
    "What is the concept of Self-RAG?",
    "How does Self-RAG handle hallucinations?",
    "What is the weather in Tokyo?" # Out-of-domain to prove CRAG works
]

# The ideal, ground-truth answers (Written by a human)
ground_truths = [
    "Self-RAG is a framework that retrieves relevant passages on-demand, and uses self-reflection to critique and select the best outputs.",
    "It uses a self-reflection mechanism with critic models to evaluate if the generated text is grounded in the retrieved facts.",
    "I do not have real-time weather information in my local database, but a web search shows the current weather in Tokyo." 
]

# --- 3. Generate Answers using SentinelRAG ---
print("Generating answers via SentinelRAG...")
agentic_answers = []
for q in eval_questions:
    # Bypass cache for accurate generation testing
    ans = process_query(q)
    agentic_answers.append(ans)

# --- 4. Prepare Ragas Dataset ---
# Note: In a real Naive RAG test, you would generate a second list called 'naive_answers' and evaluate both.
data = {
    "question": eval_questions,
    "answer": agentic_answers,
    "ground_truth": ground_truths,
    "contexts": [["Retrieved context goes here"]] * len(eval_questions) # Simplified for test
}
dataset = Dataset.from_dict(data)

# --- 5. Run Ragas Evaluation ---
print("Running Ragas Evaluation...")
result = evaluate(
    dataset=dataset,
    metrics=[faithfulness, answer_relevancy],
    llm=ragas_llm,
    embeddings=ragas_emb
)

df = result.to_pandas()
print("\n=== EVALUATION RESULTS ===")
print(df[["question", "faithfulness", "answer_relevancy"]])